Guias usadas:
- Creación, conexión con cliente Supabase y extracción de data https://supabase.com/docs/reference/python/initializing

## Previamente en SQL EDITOR - Supabase

**Consideraciones** para hacer en el proyecto de Supabase. En la sección de SQL Editor. <p>
- Crear vistas en public para que apunten a las tablas en el esquema 'raw'.(sql editor)
```
CREATE VIEW public.customers_raw as
SELECT * FROM raw.customers_raw;

CREATE VIEW public.products_raw as
SELECT * FROM raw.products_raw;

CREATE VIEW public.orders_raw as
SELECT * FROM raw.orders_raw;
```

- Generación de Columnas de products_raw sin categorias. Se copia el resultado en un string *products_column*.
```
SELECT string_agg('"' || column_name || '"', ' , ')
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name   = 'products_raw'
  AND column_name <> 'product.categories';
  ```
- Configurar número máximo de filas que retorna por proyecto o usar paginación para traer por lotes(2da opción usada).
- Creación de esquema correspondiente
```
create schema clean;
```

## Librerias

In [19]:
import sys
sys.executable

'c:\\Users\\Angelica\\Documents\\Temporal-Carrera\\PerceivoAI\\REPOS_GITHUB\\.venv\\Scripts\\python.exe'

In [82]:
import os
from dotenv import load_dotenv
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from tqdm import tqdm
from pathlib import Path
import numpy as np

transform

In [37]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv
from tqdm import tqdm
import re
import pandas as pd
import json
from rapidfuzz import process
import numpy as np


## Funciones Supabase

In [ ]:
def extract_supabase(endpoint, esquema): # cambiar despues a endpoints(lista)
    """
    Extrae datos crudos desde una tabla en Supabase asociada a un endpoint específico.
    
    Características principales:
    - Los datos se obtienen en lotes de 1000 filas (paginación con .range).
    - Por defecto soporta hasta 5000 registros, pero se puede ampliar el rango.
    - Para el endpoint "products", se especifican las columnas exactas a consultar
      (evita traer campos innecesarios o complejos como JSON anidados).
    - Para otros endpoints, se extraen todas las columnas disponibles con '*'.
    - El proceso se detiene automáticamente cuando no existen más filas en el rango.
    - Une todos los lotes extraídos en un único DataFrame de Pandas.
    
    Input: endpoint(str)
    Nombre del endpoint a consultar (ejemplo: "products", "orders", "customers").
    
    Output: Un DataFrame con todos los registros obtenidos de la tabla `{endpoint}_raw`.
    """
    name_table= f"{endpoint}_{esquema}"
    rango= [0,1000,2000,3000,4000]
    # evaluar o almacenar mientras exista rango en la tabla si lanza el error entonces parar

    parts_table=[]

    if esquema=='raw' and endpoint=='products': # columnas obtenidas luego de query en SQL EDITOR
        select_query=' "product.id" , "product.name" , "product.page_title" , "product.description" , "product.meta_description" , "product.price" , "product.cost_per_item" , "product.compare_at_price" , "product.weight" , "product.stock" , "product.stock_unlimited" , "product.stock_threshold" , "product.stock_notification" , "product.sku" , "product.brand" , "product.barcode" , "product.featured" , "product.reviews_enabled" , "product.status" , "product.shipping_required" , "product.type" , "product.days_to_expire" , "product.created_at" , "product.updated_at" , "product.package_format" , "product.length" , "product.width" , "product.height" , "product.diameter" , "product.google_product_category" , "product.images" , "product.variants" , "product.fields" , "product.permalink" , "product.discount" , "product.currency" '
        # select_query=products_column
    else:
        select_query="*"

    # Obtener data de Supabase por partes
    for valor in rango:
        table_chunk = ( 
        supabase.table(name_table)
        .select(select_query)
        .range(valor,valor+999) # 0,999 , lo mismo= rango[ind]
        .execute()
            )
        if len(table_chunk.data): # si existe la tabla en ese rango
            parts_table.append(table_chunk.data)
        else:
            break

    # Juntar todas las listas en una sola lista
    completed_table =[]
    for chunk in parts_table:
        completed_table+= chunk
    
    print(f'✅ Extracción correcta realizada para {name_table}')
    print(f"Filas: {len(completed_table)}\n")

    return pd.DataFrame(completed_table)

Las Funciones: *conect2supabase()* y *insert2supabase()* <br>
Son las mismas implementadas en [pruebas_extract.ipynb]()

In [ ]:
def conect2supabase():
    # Fetch variables
    USER = os.getenv("user")
    PASSWORD = os.getenv("password")
    HOST = os.getenv("host")
    PORT = os.getenv("port")
    DBNAME = os.getenv("dbname")

    # Construct the SQLAlchemy connection string
    DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

    # Create the SQLAlchemy engine
    engine = create_engine(DATABASE_URL)
    print('✅ Supabase: Conexión Exitosa')
    return engine

In [ ]:
def insert2supabase(engine, esquema, endpoint, df):
    table_name = f"{endpoint}_{esquema}"
    full_table_name = f"{esquema}.{table_name}"

    try:
        inspector = inspect(engine)

        # Verificar si la tabla ya existe en el esquema
        table_exists = inspector.has_table(table_name, schema=esquema)

        #☑️ SI no existe, Crear tabla desde cero
        if not table_exists:
            df.to_sql(
                table_name,
                con=engine,
                schema=esquema,
                if_exists="replace",  # replace crea la tabla si no existe
                index=False
            )
            print(f"💾 Tabla {full_table_name} creada con {len(df)} filas.")
            return

        #☑️ SI la tabla existe, se añade las nuevas filas

        # 1️⃣ Obtiene las columnas de la tabla en Supabase
        with engine.connect() as conn:
            query=f""" 
                SELECT column_name FROM information_schema.columns
                WHERE table_schema = '{esquema}'
                AND table_name = '{table_name}';
            """
            columnas_actuales= pd.read_sql(query, conn)['column_name'].tolist()

        # Filtrar df para no añadir columnas nuevas
        df= df[[c for c in df.columns if c in columnas_actuales]]


        # 2️⃣ Si la tabla ya existe, traer IDs existentes
        if esquema=='raw':
            id_name= f"{endpoint[:-1]}.id" # tengo que limpiar la ultima letra
        elif esquema=='clean':
            ID_MAP = {
                "products": "id_producto",
                "customers": "id_cliente",
                "orders": "id_orden",
                "orders_products": "id_orden"  # o (id_orden, id_producto) si necesitas PK compuesta
            }
            id_name=ID_MAP.get(endpoint)
    
        with engine.connect() as conn:
            query = f'SELECT "{id_name}" FROM {full_table_name}'
            existing_ids = pd.read_sql( query, conn )[id_name].tolist()

        # 3️⃣ Filtrar solo las filas nuevas
        df_new = df[~df[id_name].isin(existing_ids)]

        if not df_new.empty:
            df_new.to_sql(
                table_name,
                con=engine,
                schema=esquema,
                if_exists="append",
                index=False
            )
            print(f"💾 {len(df_new)} nuevas filas insertadas en {full_table_name}.")
        else:
            print(f"ℹ️ No hay nuevas filas para insertar en {full_table_name}.")

    except Exception as e:
        print(f"❌ Error insertando en {full_table_name}: {e}")


## Funciones Transform

In [39]:
def normalizar_texto_localidad(texto ):
    """
    Normaliza nombres de ciudades o municipalidades.
    
    - texto: string a normalizar
    - reemplazos: diccionario de equivalencias
    """
    reemplazos= {
    "santiago centro": "Santiago",
    "santiago de chile": "Santiago",
    "santiago  ": "Santiago",
    "ssntiago": "Santiago",
    "santiago": "Santiago",
    "SANTIAGO": "Santiago",
    "vina del mar": "Viña del Mar",
    "viña del mar ": "Viña del Mar",
    "valparaiso": "Valparaíso",
    "valparaiso ": "Valparaíso",
    "puerto montt": "Puerto Montt",
    "punta arenas": "Punta Arenas",
    "chillan": "Chillán",
    "chillan viejo": "Chillán Viejo",
    "temuco ": "Temuco",
    "temuco": "Temuco",
    "arica": "Arica",
    "antofagasta": "Antofagasta",
    "rancagua": "Rancagua",
    "curico": "Curicó",
    "san pedro de la paz": "San Pedro de la Paz",
    "los angeles": "Los Ángeles",
    "los andes": "Los Andes",
    "la serena": "La Serena",
    "valdivia": "Valdivia",
    "providencia ": "Providencia",
    "san miguel ": "San Miguel",
    "nunoa": "Ñuñoa",
    "penalolen": "Peñalolén",
    "penaflor": "Peñaflor",
    "talcahuano ": "Talcahuano",
    "concepcion": "Concepción"
    }

    if pd.isna(texto):
        return "Sin especificar"
    
    # --- 1. Limpieza básica
    t = texto.strip().lower()
    
    # --- 2. Diccionario de equivalencias conocidas
    if t in reemplazos:
        return reemplazos[t]
    
    # --- 3. Correcciones genéricas
    t = re.sub(r'\s+', ' ', t)   # quitar dobles espacios
    t = t.title()                # Capitalización estándar
    
    # --- 4. Casos inválidos genéricos
    if t in ["Asd", "Chile", "Guamuchil", "Sin Especificar"]:
        return "Sin especificar"
    
    return t


In [4]:
def transform_customers(df_raw):
    """
    Recibe el df raw de customers y devuelve un df limpio (customers_clean).
    
    Pasos aplicados:
    1. Selección de columnas relevantes
    2. Parseo de direcciones (ciudad, municipalidad, pais)
    3. Renombrado con prefijo 'cliente_'
    4. Tratamiento de nulos y creación de flags
    5. Generación de columna categórica para accepts_marketing
    """
    
    # --- 1. Selección de columnas relevantes
    columnas_filtro = [
        'customer.id',
        'customer.fullname',
        'customer.status',
        'customer.accepts_marketing',
        'customer.shipping_addresses'
    ]
    df = df_raw[columnas_filtro].copy()

    # --- Renombrado
    df = df.rename(columns={
        'customer.id': 'id_cliente',
        'customer.fullname': 'cliente_nombre',
        'customer.status':'status',
        'customer.accepts_marketing': 'acepta_marketing',
        'customer.shipping_addresses': 'direccion_envio'
    })

    
    # --- 2. Parseo de direcciones
    def parse_direcciones(fila):
        try:
            fila_parseada = json.loads(fila)
            if not fila_parseada:  # si está vacío
                return pd.Series([None, None, 'Chile'])
            
            dicc = fila_parseada[0]  # primer elemento (diccionario)
            return pd.Series([
                dicc.get('city'),
                dicc.get('municipality'),
                'Chile'
            ])
        except Exception:
            return pd.Series([None, None, 'Chile'])
    
    df[['cliente_ciudad', 'cliente_municipalidad', 'cliente_pais']] = (
        df['direccion_envio'].apply(parse_direcciones)
    )

    df['cliente_ciudad'] = df['cliente_ciudad'].apply(lambda x: normalizar_texto_localidad(x))


    df.drop(columns=['direccion_envio'], inplace=True)
    
    # --- 3. Renombrado de columnas
    # df.columns = [col.replace('customer.', 'cliente_') for col in df.columns]
    
    # --- 4. Tratamiento de nulos
    # Flags de missing

    # return df.columns
    for col in ['cliente_ciudad', 'cliente_municipalidad', 'acepta_marketing']:
        df[f'{col}_missing'] = (df[col].isna() | (df[col]== 'Sin especificar') ).astype(int)
    
    # Columnas categóricas → reemplazo por "Sin especificar"
    cols_categoricas = ['status', 'cliente_ciudad', 'cliente_municipalidad', 'cliente_pais']
    df[cols_categoricas] = df[cols_categoricas].fillna('Sin especificar')

    df['acepta_marketing'] = df['acepta_marketing'].fillna(False)
    
    # --- 5. Versión categórica de accepts_marketing
    df['marketing_cat'] = df['acepta_marketing'].map({
        True: 'Si',
        False: 'No'
    })
    
    # Resultado final
    customers_clean = df.copy()
    return customers_clean

In [5]:
def normalizar_marca(df, col="marca"): 
    """
    Normaliza la columna de marcas de productos:
    - Convierte valores nulos a 'Sin especificar'
    - Corrige errores comunes de tipeo con fuzzy matching
    """

    # ✅ Marcas más comunes
    # devuelve la lista con solo valores reales de marcas, verifica que esas marcas realmente existan, si hay Sony y SONY, escoje solo uno Sony y de manera similares, si hay variantes de una misma marca. 
    # de igual manera incluye marcas comunes que no estén en la lista
    marcas_validas = ['Accsoon', 'Benro', 'Blackmagic Design', 'Blik', 'Boya', 'Canon',
    'DJI', 'Feelworld', 'Fujifilm', 'Godox', 'Hebikuo', 'Hercules',
    'Hollyland', 'Hosa Technology', 'Iluminus', 'Joby', 'KyF', 'Lexar',
    'Lowepro', 'Maono', 'Manfrotto', 'Nanlite', 'Neewer', 'NewLine', 
    'Nikon', 'Nisi', 'NiceFoto', 'Olympus', 'Panasonic', 'Pentax',
    'Rode', 'Samsung', 'SanDisk', 'Saramonic', 'Seagate', 'Seetec',
    'Sigma', 'Sin especificar', 'Sirui', 'Sony', 'Tamron','Tascam', 'Tenba',
    'Tether Tools', 'Triopo', 'Ulanzi', 'Vijim', 'Visico', 'Viltrox',
    'Zhiyun', 'Zoom']


    # ✅ Función de limpieza
    def limpiar_empresa(x):
        if pd.isna(x):
            return "Sin especificar"
        x = str(x).strip()

        # si el valor parece numérico → error → lo tratamos como desconocido
        if x.replace(".", "").isdigit():
            return "Sin especificar"

        return x

    # ✅ Función de normalización con fuzzy
    def normalizar(x):
        x = limpiar_empresa(x)

        mejor = process.extractOne(x, marcas_validas, score_cutoff=30)
        return mejor[0] if mejor else "Sin especificar"

    # ✅ Aplicar la transformación
    df["marca"] = df[col].apply(normalizar)

    return df
#probar para una prueba producotscopy= productos_clean.copy()

In [6]:
def transform_products(df):
    columnas_filtro= ['product.id',
    'product.name',
    'product.price',
    'product.stock',
    'product.stock_threshold',
    'product.stock_notification',
    'product.brand',
    'product.reviews_enabled',
    'product.status',
    'product.created_at',
    'product.updated_at',
    'product.currency']

    # --- 1. Filtro de columnas
    df = df[columnas_filtro].copy()
    
    # --- Renombrado
    df = df.rename(columns={
        'product.id':'id_producto',
        'product.name':'descripcion',
        'product.price':'precio',
        'product.stock': 'stock',
        'product.stock_threshold': 'umbral_stock',
        'product.stock_notification': 'notificacion_stock',
        'product.brand': 'marca',
        'product.reviews_enabled': 'reseñas_habilitadas',
        'product.status': 'estado',
        'product.created_at': 'fecha_creacion',
        'product.updated_at': 'fecha_actualizacion',
        'product.currency': 'moneda'
    })
    # ---
    df =normalizar_marca(df)

    # --- 3. Relleno de categóricas
    cols_categoricas = ['descripcion', 'marca', 'estado', 'moneda']
    df[cols_categoricas] = df[cols_categoricas].fillna('Sin especificar')

    # --- 3. Manejo de fechas
    for col in ['fecha_creacion', 'fecha_actualizacion']:
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.date  # solo YYYY-MM-DD

    # --- 4. Manejo de precio
    df['precio']=df['precio'].fillna(0)

    # Resultado final
    df_clean = df.copy()
    return df_clean



In [7]:
def normalizar_empresas_envio(df, col="empresa_envio"): #empresa_envio
    """
    Normaliza la columna de empresas de envío:
    - Convierte valores numéricos a 'Sin especificar'
    - Corrige errores comunes de tipeo con fuzzy matching
    - Devuelve una nueva columna 'empresa_envio_normalizada'
    """

    # ✅ Catálogo oficial de empresas de envío
    empresas_validas = [
        "Envío Express",
        "Blue Express",
        "BluExpress Express",
        "BluExpress Priority",
        "Estándar a domicilio",
        "Estándar a sucursal",
        "Prioritario",
        "Prioritario a domicilio",
        "Prioritario a sucursal",
        "Correo Ordinario",
        "Sin especificar"
    ]

    # ✅ Función de limpieza
    def limpiar_empresa(x):
        if pd.isna(x):
            return "Sin especificar"
        x = str(x).strip()

        # si el valor parece numérico → error → lo tratamos como desconocido
        if x.replace(".", "").isdigit():
            return "Sin especificar"

        return x

    # ✅ Función de normalización con fuzzy
    def normalizar_empresa(x):
        x = limpiar_empresa(x)

        mejor = process.extractOne(x, empresas_validas, score_cutoff=30)
        return mejor[0] if mejor else "Sin especificar"

    # ✅ Aplicar la transformación
    df["empresa_envio"] = df[col].apply(normalizar_empresa)

    return df


In [ ]:
def transform_orders(df_raw):
    """
    Recibe el df raw de orders y devuelve orders_clean.
    Aplica: selección, renombrado, parseo, mapeo, tratamiento de nulos y tipos.
    """
    
    columnas_filtro = [
        'order.id',
        'order.created_at',
        'order.currency',
        'order.total',
        'order.fulfillment_status',
        'order.shipping_method_name',
        'order.customer.id',
        'order.shipping_address.region',
        'order.shipping_address.country',
        'order.shipping_address.municipality',
        'order.products',
        'order.status',
        'order.shipment_status'
    ]
    
    df = df_raw[columnas_filtro].copy()
    
    # --- Renombrado
    df = df.rename(columns={
        'order.id': 'id_orden',
        'order.created_at': 'fecha_creacion',
        'order.currency': 'moneda',
        'order.total': 'precio_total',
        'order.fulfillment_status': 'estado_cumplimiento',
        'order.shipping_method_name': 'empresa_envio',
        'order.customer.id': 'id_cliente',
        'order.shipping_address.region': 'region_envio',
        'order.shipping_address.country': 'pais_envio',
        'order.shipping_address.municipality': 'municipalidad_envio',
        'order.products': 'productos',
        'order.status': 'estado_orden',
        'order.shipment_status': 'estado_envio'
    })
    
    # --- Tipos y nulos
    df['id_orden'] = pd.to_numeric(df['id_orden'], errors='coerce')
    df = df.drop_duplicates(subset='id_orden')
    
    df['fecha_creacion'] = pd.to_datetime(df['fecha_creacion'], errors='coerce').dt.date
    # df['fecha_completacion'] = pd.to_datetime(df['fecha_completacion'], errors='coerce').dt.date
    
    df['moneda'] = df['moneda'].fillna('CLP')
    df['precio_total'] = pd.to_numeric(df['precio_total'], errors='coerce').fillna(0).astype(int)
    

    df['id_cliente'] = df['id_cliente'].fillna(0).astype(int) #generar una columna si id_cliente tiene valor diferente a 0 'Registrado' sino 'No registrado'
    df['estado_cliente']= np.where(df['id_cliente'] != 0, 'Registrado', 'No Registrado')
    df['pais_envio'] = df['pais_envio'].fillna('Chile')
    

    # --- Parsear productos (extraer ids de productos)
    def extraer_ids(productos):
        try:
            items = json.loads(productos) if isinstance(productos, str) else productos
            if isinstance(items, list):
                return ";".join(str(p.get("id")) for p in items if p.get("id"))
        except Exception:
            return None
    df['productos_ids'] = df['productos'].apply(extraer_ids)
    df.drop(columns=['productos'], inplace=True)

    df['estado_cumplimiento'] = df['estado_cumplimiento'].map({
        'unfulfilled': 'No cumplido',
        'fulfilled': 'Cumplido' })
    
    df['estado_orden'] = df['estado_orden'].map({
        "Abandoned": "Abandonada",
        "Paid": "Pagada",
        "Canceled": "Cancelada",
        "Created": "Creada",
        "Pending Payment": "Pendiente de pago" })
    
    df['estado_envio'] = df['estado_envio'].map({
        "No Procesado": "No procesado",
        "Entregado": "Entregado",
        "Solicitado": "Solicitado",
        "No Aplicable": "No aplicable",
        "En Tránsito": "En tránsito"  })

    # nulos
    cols = ['estado_cumplimiento', 'region_envio', 'municipalidad_envio','estado_orden','estado_envio'] #empresas_envio
    df[cols] = df[cols].fillna('Sin especificar')

    # normaliza empresas de envio
    df = normalizar_empresas_envio(df)
    
    
    return df


In [10]:
def create_orders_products(df_orders, col_order="id_orden", col_products="productos_ids"):
    """
    Crea una tabla de relación orders_products a partir de un dataframe de órdenes.
    df_orders: DataFrame con al menos las columnas id_order y id_producto (string con ';' separados).
    col_order: nombre de la columna con el id de la orden.
    col_products: nombre de la columna con los ids de productos (str separados por ';').
    """
    
    # copiar para no alterar df original
    df_rel = df_orders[[col_order, col_products]].copy()
    
    # separar productos en listas
    df_rel[col_products] = df_rel[col_products].astype(str).str.split(";")
    
    # "explotar" listas en filas
    df_rel = df_rel.explode(col_products).reset_index(drop=True)
    
    # convertir id_producto a entero si es posible
    df_rel[col_products] = pd.to_numeric(df_rel[col_products], errors="coerce").astype("Int64")
    
    # renombrar columnas para consistencia
    df_rel = df_rel.rename(columns={
        col_order: "id_orden",
        col_products: "id_producto"
    })
    
    return df_rel


## FLUJO DE EJECUCIÓN - (data/raw → procesado)

#### 0. Conexión Cliente Supabase

In [11]:

load_dotenv()
url= os.environ.get("SUPABASE_URL")
key= os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)


#### 1. Ejecución de extracción(data_raw)

In [ ]:

endpoints=['products', 'customers', 'orders']
esquema='raw'
df_raw= {}
for endpoint in tqdm(endpoints):
    df_raw[f'{endpoint}_raw']= extract_supabase(endpoint=endpoint, esquema=esquema) 


 33%|███▎      | 1/3 [00:05<00:11,  5.70s/it]

✅ Extracción correcta realizada para products_raw
Filas: 1339



 67%|██████▋   | 2/3 [00:07<00:03,  3.25s/it]

✅ Extracción correcta realizada para customers_raw
Filas: 1851



100%|██████████| 3/3 [00:11<00:00,  3.79s/it]

✅ Extracción correcta realizada para orders_raw
Filas: 2333



In [35]:
df_raw.keys()

dict_keys(['products_raw', 'customers_raw', 'orders_raw'])

#### 2. Aplica funciones realizadas para cada tabla (customers, products, orders) y se generan data_clean

In [ ]:
endpoints= [ "products","customers","orders"]
dataframes_clean ={}
for endpoint in endpoints:
    funcion= globals().get(f"transform_{endpoint}")
    dataframes_clean[f"{endpoint}_clean"] = funcion(df_raw[f'{endpoint}_raw'])  #ej: transform_orders(df_raw['orders_raw'])

dataframes_clean["orders_products_clean"] = create_orders_products(dataframes_clean["orders_clean"])


C:\Users\Angelica\AppData\Local\Temp\ipykernel_22876\582853927.py:72: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['acepta_marketing'] = df['acepta_marketing'].fillna(False)


In [134]:
dataframes_clean.keys()

dict_keys(['products_clean', 'customers_clean', 'orders_clean', 'orders_products_clean'])

In [135]:
# Verifica que existan id unicos
for key in dataframes_clean.keys():
    print('Tabla: ',key)
    df=dataframes_clean[f'{key}']
    id_column= df.columns[0] # ej: id_producto
    if len(df) == df[id_column].nunique():
        print(f'  ID: {id_column} SÍ contiene valores únicos')
    else:
        print(f'  ID {id_column} NO contiene valores únicos')


Tabla:  products_clean
  ID: id_producto SÍ contiene valores únicos
Tabla:  customers_clean
  ID: id_cliente SÍ contiene valores únicos
Tabla:  orders_clean
  ID: id_orden SÍ contiene valores únicos
Tabla:  orders_products_clean
  ID id_orden NO contiene valores únicos


In [ ]:
dataframes_clean['products_clean'].head(2)

,id_producto,descripcion,precio,stock,umbral_stock,notificacion_stock,marca,reseñas_habilitadas,estado,fecha_creacion,fecha_actualizacion,moneda
0,27766653,Cámara Canon Mirrorless EOS R8 Body,1999990,1,0,True,Canon,True,available,2024-11-23,2025-08-12,CLP
1,27766654,Cámara Canon Mirrorless EOS R6 MKII Body,2989990,1,0,True,Canon,True,available,2024-11-23,2025-08-05,CLP


In [ ]:
dataframes_clean['customers_clean'].head(2)

,id_cliente,cliente_nombre,status,acepta_marketing,cliente_ciudad,cliente_municipalidad,cliente_pais,cliente_ciudad_missing,cliente_municipalidad_missing,acepta_marketing_missing,marketing_cat
0,17276830,Francisca Fuenzalida,approved,False,La Serena,La Serena,Chile,0,0,0,No
1,15462049,Oscar Herrera,approved,False,Marga Marga,Quilpué,Chile,0,0,0,No


In [ ]:

dataframes_clean['orders_clean'].head(2)

,id_orden,fecha_creacion,moneda,precio_total,estado_cumplimiento,empresa_envio,id_cliente,region_envio,pais_envio,municipalidad_envio,estado_orden,estado_envio,estado_cliente,productos_ids
0,2679,2025-08-27,CLP,167990,No cumplido,Envío Express,0,O'Higgins,Chile,Marchihue,Abandonada,No procesado,No Registrado,28363657
1,2678,2025-08-27,CLP,839990,No cumplido,Envío Express,17569269,Región Metropolitana,Chile,Vitacura,Pagada,No procesado,Registrado,27766673


**Revisar**:
- chequeo generar que las data_clean esten correctas, valores adecuados, etc
- integrarlas todas las funciones en un archivo .py 'transform' 
- probar flujo correcto
- cargar los nuevos datasets a supabase (data_clean) cómo gestionas valores adicionales en supabase(usar misma lógica para verificar id existentes)

## FLUJO DE EJECUCIÓN - LOAD
(procesado → Supabase)

Se carga a supabase.
Previamente en Supabase: crear nuevo esquema(configurar nuevo esquema)

In [ ]:
# necesito subir a supabase
from sqlalchemy import create_engine, inspect
import os
login =  os.getenv('JUMPSELLER_LOGIN')
authtoken =  os.getenv('JUMPSELLER_AUTHTOKEN')

engine_supabase= conect2supabase()

✅ Supabase: Conexión Exitosa


In [125]:
esquema= 'clean'
endpoints= [ "products","customers","orders","orders_products"]

# for key, df in tqdm(dataframes_clean.items(), desc= 'Insertando en Supabase'):
for endpoint in tqdm(endpoints, desc='Insertando en Supabase'):
    df= dataframes_clean[f'{endpoint}_clean']
    insert2supabase(engine_supabase, esquema, endpoint, df)

Insertando en Supabase:  25%|██▌       | 1/4 [00:01<00:04,  1.37s/it]

💾 Tabla clean.products_clean creada con 1339 filas.


Insertando en Supabase:  50%|█████     | 2/4 [00:02<00:02,  1.20s/it]

💾 Tabla clean.customers_clean creada con 1851 filas.


Insertando en Supabase:  75%|███████▌  | 3/4 [00:03<00:01,  1.28s/it]

💾 Tabla clean.orders_clean creada con 2333 filas.


Insertando en Supabase: 100%|██████████| 4/4 [00:05<00:00,  1.26s/it]

💾 Tabla clean.orders_products_clean creada con 2973 filas.
